Importation des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pingouin as pg
import seaborn as sns
import statsmodels.stats.diagnostic as sms
from sklearn.linear_model import LinearRegression
from statsmodels.stats.stattools import durbin_watson
from scipy.stats import gaussian_kde
from scipy.stats import genpareto, chi2, anderson_ksamp, kstest
from scipy.stats import skew, kurtosis
import scipy.stats as stats
from ipywidgets import interact, FloatSlider
import plotly.graph_objects as go
from scipy.stats import genpareto, expon

Importation des données

In [ ]:
data = pd.read_excel("Données.xlsx")
data

In [ ]:


# 1. Sélectionner les colonnes à analyser
colonnes_a_analyser = ["Montant actualisés"]

# 2. Calcul des statistiques descriptives de base (Pandas)
analyse_descriptive = data[colonnes_a_analyser].describe().loc[['count', 'mean', 'std', 'min', '50%', 'max']]

# Renommer l'index pour correspondre exactement à vos termes
analyse_descriptive = analyse_descriptive.rename(
    index={
        "count": "Nombre d'observations",
        "mean": "Moyenne",
        "std": "Écart-type",
        "min": "Minimum",
        "50%": "Médiane",
        "max": "Maximum",
    }
)

# 3. Calcul du Skewness (Coefficient d'asymétrie)
# Par défaut, Pandas calcule le skewness ajusté du biais informatique
skewness = data[colonnes_a_analyser].skew()

# 5. Intégration du Skewness au tableau final
analyse_descriptive.loc["Skewness"] = skewness

# 6. Affichage du résultat
analyse_descriptive

Mean Excess Plot

In [ ]:
def plot_dynamic_mean_excess(data, column_name):
    clean_data = data[column_name].dropna()
    clean_data = np.sort(clean_data[clean_data > 0].values)
    
    @interact(u=FloatSlider(min=clean_data.min(), max=clean_data.max()*0.8, step=1000000, value=clean_data.mean()))
    def update_plot(u):
        # Calcul du Mean Excess
        thresholds = clean_data[clean_data < clean_data.max() * 0.9]
        mean_excess = [np.mean(clean_data[clean_data > t] - t) for t in thresholds]
        
        plt.figure(figsize=(10, 5))
        plt.plot(thresholds, mean_excess, 'b-', alpha=0.6)
        plt.axvline(x=u, color='red', linestyle='--', label=f'Seuil testé : {u:,.0f}')
        
        # Focus sur l'excès au seuil choisi
        current_excess = np.mean(clean_data[clean_data > u] - u)
        plt.plot(u, current_excess, 'ro')
        
        plt.title("Mean Excess Plot Interactif")
        plt.xlabel("Seuil (u)")
        plt.ylabel("e(u)")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

plot_dynamic_mean_excess(data, 'Montant actualisés')

Threshold Stability Plot

In [ ]:
# 1. Préparation des données (on prend tout ce qui est positif pour tester plusieurs seuils)
data_all = data.loc[data['Montant actualisés'] > 0, 'Montant actualisés'].values

def threshold_stability_plot(data, thresholds):
    shape_params = []
    scale_params_modified = [] # sigma* = sigma - xi*u
    
    for u in thresholds:
        excesses = data[data > u] - u
        if len(excesses) < 10: # Minimum de données pour un fit stable
            shape_params.append(np.nan)
            scale_params_modified.append(np.nan)
            continue
            
        # Fit de la GPD (floc=0 car on modélise les excès)
        xi, loc, sigma = genpareto.fit(excesses, floc=0)
        
        shape_params.append(xi)
        # On utilise le paramètre d'échelle modifié pour la stabilité
        scale_params_modified.append(sigma - xi * u)

    # --- GRAPHIQUES CÔTE À CÔTE ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # Graphique du paramètre de forme (xi)
    ax1.plot(thresholds, shape_params, 'o-', color='blue', markersize=4)
    ax1.axhline(y=0.402, color='red', linestyle='--', label='Cible $\\xi=0.402$')
    ax1.set_title('Stabilité du paramètre de forme $\\xi$')
    ax1.set_xlabel('Seuil $u$')
    ax1.set_ylabel('Estimation de $\\xi$')
    ax1.grid(True, alpha=0.3)

    # Graphique du paramètre d'échelle modifié (sigma*)
    ax2.plot(thresholds, scale_params_modified, 'o-', color='green', markersize=4)
    ax2.set_title('Stabilité du paramètre d\'échelle modifié $\sigma^*$')
    ax2.set_xlabel('Seuil $u$')
    ax2.set_ylabel('$\sigma - \\xi u$')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# 2. Définition de la plage de seuils à tester
# On teste par exemple de 100M à 800M par pas de 20M
seuils_test = np.arange(100_000_000, 800_000_000, 20_000_000)

threshold_stability_plot(data_all, seuils_test)

Comparaison des quantiles empiriques avec ce de la loi exponentielle

In [ ]:
def plot_qq_exponential(data, column_name):
    # 1. Nettoyage et préparation des données
    # On ne travaille que sur ce qui dépasse le seuil u (loi des excès)
    # Si tu n'as pas encore fixé u, on prend toutes les données positives
    clean_data = data[column_name].dropna()
    clean_data = clean_data[clean_data > 0].sort_values()

    # 2. Création du graphique
    plt.figure(figsize=(8, 8))
    
    # stats.probplot calcule les quantiles théoriques d'une loi exponentielle
    # et les compare à tes données
    res = stats.probplot(clean_data, dist="expon", plot=plt)
    
    # 3. Personnalisation
    plt.title(f'QQ-Plot : Données réelles vs Loi Exponentielle\n({column_name})', fontsize=12)
    plt.xlabel('Quantiles théoriques (Loi Exponentielle)', fontsize=10)
    plt.ylabel('Quantiles observés (Sinistres actualisés)', fontsize=10)
    plt.grid(True, which="both", ls="-", alpha=0.5)
    
    # Affichage
    print("Affichage du QQ-Plot... Si la courbe monte plus vite que la droite, la queue est lourde.")
    plt.show()

# Utilisation :
plot_qq_exponential(data, 'Montant actualisés')

Estimation de la loi GPD

In [ ]:
def modeliser_gpd(data, column_name, threshold):
    # 1. Extraction des excès (Y = X - u)
    # On ne garde que les montants strictement supérieurs au seuil choisi
    excesses = data[data[column_name] > threshold][column_name] - threshold
    
    if len(excesses) < 10:
        print("Erreur : Trop peu de données au-dessus du seuil pour un ajustement.")
        return None

    # 2. Ajustement de la loi (Estimation des paramètres)
    # floc=0 car la GPD des excès commence théoriquement à zéro
    xi, loc, sigma = genpareto.fit(excesses, floc=0)
    
    print("-" * 30)
    print(f"PARAMÈTRES GPD ESTIMÉS (u = {threshold})")
    print("-" * 30)
    print(f"Indice de queue (xi)   : {xi:.4f}")
    print(f"Paramètre d'échelle (sigma) : {sigma:.4f}")
    print(f"Nombre d'excès utilisés : {len(excesses)}")
    print("-" * 30)

    # 3. Visualisation de l'ajustement
    plt.figure(figsize=(10, 6))
    
    # Histogramme des données réelles (en densité)
    plt.hist(excesses, bins=20, density=True, alpha=0.3, color='gray', label='Excès observés')
    
    # Courbe de la loi GPD thérorique
    x = np.linspace(0, excesses.max(), 200)
    pdf_gpd = genpareto.pdf(x, xi, loc, sigma)
    plt.plot(x, pdf_gpd, 'r-', lw=2, label=f'GPD ajustée (xi={xi:.2f})')
    
    plt.title('Modélisation de la Sévérité des Excès (GPD)')
    plt.xlabel('Montant des excès (Sinistre - Seuil)')
    plt.ylabel('Densité')
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

    return xi, sigma

# Utilisation :
xi_estime, sigma_estime = modeliser_gpd(data, 'Montant actualisés', 600000000)

Comparaison des quantiles empiriques avec ceux de la GPD

In [ ]:
# --- 1. LES DONNÉES ET PARAMÈTRES ---

xi = 0.4003
sigma = 1_748_298_084.89 
u = 600_000_000

# Simulation de données réelles pour l'exemple
np.random.seed(42)
donnees_reelles = genpareto.rvs(xi, loc=u, scale=sigma, size=500)

# --- 2. PRÉPARATION DU QQ-PLOT ---
# On trie les données réelles par ordre croissant
donnees_triees = np.sort(donnees_reelles)
n = len(donnees_triees)

# Calcul des probabilités empiriques (formule de Weibull : i / (n+1))
probabilites = np.arange(1, n + 1) / (n + 1)

# Calcul des quantiles théoriques de la GPD pour ces probabilités
quantiles_theoriques = genpareto.ppf(probabilites, xi, loc=u, scale=sigma)

# --- 3. GÉNÉRATION DU GRAPHIQUE ---
plt.figure(figsize=(8, 6))
plt.scatter(quantiles_theoriques, donnees_triees, color='blue', alpha=0.5, label='Données réelles vs GPD')

# Ajout de la ligne identité (y = x)
lims = [
    np.min([quantiles_theoriques.min(), donnees_triees.min()]),
    np.max([quantiles_theoriques.max(), donnees_triees.max()])
]
plt.plot(lims, lims, color='red', linestyle='--', label='Alignement parfait')

# Cosmétique pour le mémoire
plt.title(f'QQ-Plot : Distribution des Sinistres vs Loi GPD (xi={xi})', fontsize=12)
plt.xlabel('Quantiles Théoriques (Modèle GPD)', fontsize=10)
plt.ylabel('Quantiles Observés (Données Réelles)', fontsize=10)
plt.legend()
plt.grid(True, which='both', linestyle='--', alpha=0.5)



plt.show()

Comparaison des probilité empiriques et celles de la GPD

In [ ]:
# 1. Préparation des données
# On récupère les excès par rapport au seuil u
u = 600_000_000  # Remplacer par votre vrai seuil u
exces = data.loc[data['Montant actualisés'] > u, 'Montant actualisés'].values - u

# 2. Tri des données par ordre croissant
donnees_triees = np.sort(exces)
n = len(donnees_triees)

# 3. Estimation des paramètres de la GPD par Maximum de Vraisemblance (MLE)
# c = paramètre de forme (gamma), loc = paramètre de position, scale = échelle (sigma)
gamma_estim, loc_estim, sigma_estim = stats.genpareto.fit(donnees_triees, floc=0)

# 4. Calcul des probabilités empiriques (Fréquences cumulées cumulées)
# Le réajustement -0.5 permet d'éviter d'avoir une probabilité exacte de 1 au dernier point
prob_empiriques = (np.arange(1, n + 1) - 0.5) / n

# 5. Calcul des probabilités théoriques sous l'hypothèse de la loi GPD calibrée
prob_theoriques = stats.genpareto.cdf(donnees_triees, c=gamma_estim, loc=loc_estim, scale=sigma_estim)

# 6. Construction du P-P Plot
plt.figure(figsize=(7, 7))
plt.scatter(prob_theoriques, prob_empiriques, color='blue', edgecolor='k', alpha=0.6, label='Données observées')

# Ligne de référence (première bissectrice y = x)
plt.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=2, label='Ajustement parfait')

# Configuration des axes et titres
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.xlabel('Probabilités théoriques (Modèle GPD)', fontsize=11)
plt.ylabel('Probabilités empiriques (Données)', fontsize=11)
plt.title('P-P Plot - Ajustement à la loi de Pareto Générale (GPD)', fontsize=12, fontweight='bold')
plt.legend(loc='upper left')
plt.grid(True, linestyle=':', alpha=0.6)

# Affichage du graphique
plt.show()

# Affichage des paramètres estimés pour vérification dans votre console
print(f"--- Paramètres GPD estimés ---")
print(f"Paramètre de forme (γ) : {gamma_estim:.4f}")
print(f"Paramètre d'échelle (σ) : {sigma_estim:.4f}")

Test du rapport de vraisemblance

In [ ]:
def test_rapport_vraisemblance(data, column_name, u):
    """
    Réalise le LRT pour comparer Loi Exponentielle (H0) vs GPD (H1).
    u : le seuil (threshold) que tu as déterminé.
    """
    # 1. Extraction des excès
    excesses = data[data[column_name] > u][column_name] - u
    n = len(excesses)
    
    if n < 10:
        print(f"Échantillon trop faible ({n} excès). Le test risque d'être instable.")
    
    # 2. Modèle H1 : GPD (Paramètre xi est libre)
    # On fixe floc=0 car on travaille sur les excès
    xi_h1, loc_h1, sigma_h1 = genpareto.fit(excesses, floc=0)
    log_lik_h1 = np.sum(genpareto.logpdf(excesses, xi_h1, loc_h1, sigma_h1))
    
    # 3. Modèle H0 : Exponentielle (On force xi = 0)
    # fc=0 fige le paramètre de forme à zéro
    xi_h0, loc_h0, sigma_h0 = genpareto.fit(excesses, fc=0, floc=0)
    log_lik_h0 = np.sum(genpareto.logpdf(excesses, xi_h0, loc_h0, sigma_h0))
    
    # 4. Statistique du test (D)
    # D = 2 * (Log-Vraisemblance Modèle Complexe - Log-Vraisemblance Modèle Simple)
    d_stat = 2 * (log_lik_h1 - log_lik_h0)
    
    # 5. Calcul de la p-valeur
    # La statistique D suit une loi du Chi-deux à 1 degré de liberté (la différence de paramètres)
    p_val = chi2.sf(d_stat, df=1)
    
    # --- Affichage des résultats ---
    print(f"--- RÉSULTATS DU LRT (Seuil u = {u:,.0f}) ---")
    print(f"Nombre d'excès : {n}")
    print(f"Log-Likelihood GPD (xi={xi_h1:.3f}) : {log_lik_h1:.2f}")
    print(f"Log-Likelihood Expo (xi=0) : {log_lik_h0:.2f}")
    print(f"Statistique du test (D) : {d_stat:.4f}")
    print(f"P-valeur : {p_val:.6f}")
    
    if p_val < 0.05:
        print("=> RÉSULTAT : Significatif au seuil de 5%.")
        print("On rejette H0 : La queue de distribution est lourde (GPD nécessaire).")
    else:
        print("=> RÉSULTAT : Non significatif.")
        print("On ne peut pas rejeter H0 : Une loi exponentielle suffit.")

# Appel du test :
test_rapport_vraisemblance(data, 'Montant actualisés', 600000000)

Test Kolmogorov Smirnov

In [ ]:
# --- 1. PRÉPARATION DES DONNÉES ---
# On récupère les excès par rapport au seuil u
u = 600_000_000  # Remplacer par votre vrai seuil u
exces = data.loc[data['Montant actualisés'] > u, 'Montant actualisés'].values - u

# --- 2. ESTIMATION DES PARAMÈTRES GPD ---
# xi_est : paramètre de forme (γ)
# sigma_est : paramètre d'échelle (σ)
xi_est, loc_est, sigma_est = genpareto.fit(exces, floc=0)

# --- 3. TEST DE KOLMOGOROV-SMIRNOV (KS) ---
# ks_1samp prend en argument :
# - l'échantillon d'excès
# - la fonction de répartition (cdf) de la loi théorique choisie
# - les paramètres de cette loi sous forme de tuple (forme, position, échelle)
ks_stat, p_val = kstest(
    exces, 
    'genpareto', 
    args=(xi_est, loc_est, sigma_est)
)

# --- 4. AFFICHAGE DES RÉSULTATS ---
print("="*40)
print("     RÉSULTATS DU TEST KS (GPD)")
print("="*40)
print(f"Nombre d'observations > u : {len(exces)}")
print(f"Paramètre de forme (xi)  : {xi_est:.4f}")
print(f"Paramètre d'échelle (sigma): {sigma_est:,.2f}")
print("-"*40)
print(f"TEST DE KOLMOGOROV-SMIRNOV")
print(f"Statistique D_n          : {ks_stat:.4f}")
print(f"P-value                  : {p_val:.4f}")
print("-"*40)

if p_val > 0.05:
    print("VERDICT : L'ajustement est validé par le test KS (p-value > 5%).")
else:
    print("VERDICT : L'ajustement est rejeté par le test KS (p-value <= 5%).")
print("="*40)

Test Anderson-Darling

In [ ]:
# --- 1. PRÉPARATION DES DONNÉES ---
# On récupère les vraies valeurs numériques au-dessus du seuil u
u = 600_000_000
exces = data.loc[data['Montant actualisés'] > u, 'Montant actualisés'].values - u

# --- 2. ESTIMATION GPD ---
# xi_est : paramètre de forme (ton fameux 0.402)
# sigma_est : paramètre d'échelle
xi_est, loc_est, sigma_est = genpareto.fit(exces, floc=0)

# --- 3. TEST D'ANDERSON-DARLING ---
# On compare l'échantillon d'excès à la loi GPD théorique
# On génère un échantillon théorique large pour la comparaison
data_theorique = genpareto.rvs(xi_est, scale=sigma_est, size=10000, random_state=42)
ad_stat, crit_vals, p_val = anderson_ksamp([exces, data_theorique])

# --- 4. AFFICHAGE DES RÉSULTATS ---
print("="*40)
print("   RÉSULTATS DE L'AJUSTEMENT GPD")
print("="*40)
print(f"Nombre d'observations > u : {len(exces)}")
print(f"Paramètre de forme (xi)  : {xi_est:.4f}")
print(f"Paramètre d'échelle (sigma): {sigma_est:,.2f}")
print("-"*40)
print(f"TEST D'ANDERSON-DARLING")
print(f"Statistique A2           : {ad_stat:.4f}")
print(f"P-value                  : {p_val:.4f}")
print("-"*40)

if p_val > 0.05:
    print("VERDICT : L'ajustement est validé (p-value > 5%).")
else:
    print("VERDICT : L'ajustement est rejeté (p-value <= 5%).")
print("="*40)

Comparaison des critères AIC et BIC entre GPD et exponentielle

In [ ]:
# --- 1. PRÉPARATION DES DONNÉES ---
# On récupère les excès par rapport au seuil u
u = 600_000_000  # À ajuster avec votre vrai seuil u
exces = data.loc[data['Montant actualisés'] > u, 'Montant actualisés'].values - u
n = len(exces)

# --- 2. AJUSTEMENT ET CRITÈRES POUR LA LOI GPD ---
# Paramètres GPD : xi (forme), loc (fixé à 0), sigma (échelle) -> k = 2 paramètres estimés
xi_gpd, loc_gpd, sigma_gpd = genpareto.fit(exces, floc=0)
log_lik_gpd = np.sum(genpareto.logpdf(exces, xi_gpd, loc=0, scale=sigma_gpd))

k_gpd = 2  # Nombre de paramètres libres (xi et sigma)
aic_gpd = 2 * k_gpd - 2 * log_lik_gpd
bic_gpd = k_gpd * np.log(n) - 2 * log_lik_gpd


# --- 3. AJUSTEMENT ET CRITÈRES POUR LA LOI EXPONENTIELLE ---
# Paramètres Exponentielle : loc (fixé à 0), scale (échelle = moyenne) -> k = 1 paramètre estimé
loc_exp, scale_exp = expon.fit(exces, floc=0)
log_lik_exp = np.sum(expon.logpdf(exces, loc=0, scale=scale_exp))

k_exp = 1  # Nombre de paramètres libres (scale)
aic_exp = 2 * k_exp - 2 * log_lik_exp
bic_exp = k_exp * np.log(n) - 2 * log_lik_exp


# --- 4. SÉLECTION ET COMPARAISON DES RÉSULTATS ---
resultats = pd.DataFrame({
    "Loi GPD": [log_lik_gpd, k_gpd, aic_gpd, bic_gpd],
    "Loi Exponentielle": [log_lik_exp, k_exp, aic_exp, bic_exp]
}, index=["Log-Vraisemblance", "Nb Paramètres (k)", "AIC", "BIC"])

# Arrondir pour un affichage propre
print("="*50)
print("   COMPARAISON DES MODÈLES D'AJUSTEMENT (AIC / BIC)")
print("="*50)
print(f"Nombre total d'observations (n) : {n}")
print("-"*50)
print(resultats.round(3))
print("-"*50)

# Règle de décision automatique
print("VERDICT SELECTION :")
if aic_gpd < aic_exp:
    print(" -> L'AIC privilégie la loi GPD.")
else:
    print(" -> L'AIC privilégie la loi Exponentielle.")

if bic_gpd < bic_exp:
    print(" -> Le BIC privilégie la loi GPD.")
else:
    print(" -> Le BIC privilégie la loi Exponentielle.")
print("="*50)

Comparaison graphique des ajustement GPD vs Exponentielle

In [ ]:
def comparer_ajustements_interactif(data, column_name, threshold):
    # 1. Préparation des excès
    excesses = data[data[column_name] > threshold][column_name] - threshold
    excesses = np.sort(excesses.dropna().values)
    
    # 2. Ajustement des modèles
    # GPD
    shape_gpd, loc_gpd, scale_gpd = genpareto.fit(excesses, floc=0)
    log_lik_gpd = np.sum(genpareto.logpdf(excesses, shape_gpd, loc_gpd, scale_gpd))
    aic_gpd = 2*2 - 2*log_lik_gpd
    
    # Exponentielle
    loc_exp, scale_exp = expon.fit(excesses, floc=0)
    log_lik_exp = np.sum(expon.logpdf(excesses, loc_exp, scale_exp))
    aic_exp = 2*1 - 2*log_lik_exp

    # 3. Création de la figure Plotly
    fig = go.Figure()

    # Histogramme des données réelles
    fig.add_trace(go.Histogram(
        x=excesses,
        histnorm='probability density',
        name='Données Réelles',
        marker_color='lightgray',
        opacity=0.5,
        nbinsx=20
    ))

    # Axe X pour les courbes théoriques
    x_range = np.linspace(0, excesses.max() * 1.3, 500)

    # Courbe GPD
    y_gpd = genpareto.pdf(x_range, shape_gpd, loc_gpd, scale_gpd)
    fig.add_trace(go.Scatter(
        x=x_range, y=y_gpd,
        mode='lines',
        name=f'GPD (AIC: {aic_gpd:.1f})',
        line=dict(color='red', width=3),
        hovertemplate='Montant: %{x:,.0f} DA<br>Densité GPD: %{y:.4e}<extra></extra>'
    ))

    # Courbe Exponentielle
    y_exp = expon.pdf(x_range, loc_exp, scale_exp)
    fig.add_trace(go.Scatter(
        x=x_range, y=y_exp,
        mode='lines',
        name=f'Exponentielle (AIC: {aic_exp:.1f})',
        line=dict(color='blue', dash='dash', width=2),
        hovertemplate='Montant: %{x:,.0f} DA<br>Densité Exp: %{y:.4e}<extra></extra>'
    ))

    # 4. Mise en page et annotations
    fig.update_layout(
        title=f"Comparaison des Modèles (Seuil u = {threshold:,.0f} DA)",
        xaxis_title="Montant des Excès (Sinistre - u)",
        yaxis_title="Densité de Probabilité",
        template="plotly_white",
        hovermode="x unified",
        legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99),
        annotations=[
            dict(
                x=0.7, y=0.5, xref="paper", yref="paper",
                text=f"<b>Paramètres GPD :</b><br>ξ (Shape) : {shape_gpd:.3f}<br>σ (Scale) : {scale_gpd:,.0f}",
                showarrow=False,
                bgcolor="white", bordercolor="black", borderpad=4
            )
        ]
    )

    # Force l'ouverture dans le navigateur par défaut
    fig.show(renderer="browser")

# Utilisation :
comparer_ajustements_interactif(data, 'Montant actualisés', 600000000)